In [ ]:
# Enhanced Dash Dashboard with User Authentication and Role-Based Views
from dash import dcc, html, dash_table, Input, Output, State
from jupyter_dash import JupyterDash
import dash_leaflet as dl
import plotly.express as px
import pandas as pd
import os
from animal_shelter import AnimalShelter

# Default MongoDB Credentials (replace or secure in real use)
USERNAME = "root"
PASSWORD = os.getenv('MONGO_PASSWORD', 'default_password')
HOST = "nv-desktop-services.apporto.com"
PORT = 32355

# Dash App Setup
app = JupyterDash("CS340 Authenticated Dashboard")

# Instantiate backend connection to verify login
shelter_auth = AnimalShelter(USERNAME, PASSWORD, HOST, PORT, 'AAC', 'animals')

app.layout = html.Div([
    html.H1("Grazioso Salvare Animal Dashboard with Login", style={'textAlign': 'center'}),
    html.Div(id='login-area', children=[
        dcc.Input(id='input-username', type='text', placeholder='Username'),
        dcc.Input(id='input-password', type='password', placeholder='Password'),
        html.Button('Login', id='login-button', n_clicks=0),
        html.Div(id='login-message')
    ]),
    html.Div(id='dashboard-area', style={'display': 'none'})  # Hidden until login
])

@app.callback(
    Output('login-message', 'children'),
    Output('dashboard-area', 'style'),
    Output('dashboard-area', 'children'),
    Input('login-button', 'n_clicks'),
    State('input-username', 'value'),
    State('input-password', 'value')
)
def process_login(n_clicks, username, password):
    if n_clicks > 0:
        role = shelter_auth.validate_user(username, password)
        if role:
            # Load dashboard content based on role
            df = pd.DataFrame.from_records(shelter_auth.read({}))
            df.drop(columns=['_id'], inplace=True)

            data_table = dash_table.DataTable(
                id='data-table',
                columns=[{"name": i, "id": i} for i in df.columns],
                data=df.to_dict('records'),
                page_size=10,
                style_table={'overflowX': 'auto'}
            )

            pie_chart = dcc.Graph(
                id='pie-chart',
                figure=px.pie(df, names='breed')
            )

            map_placeholder = dl.Map(
                id='map-id',
                style={'width': '1000px', 'height': '500px'},
                center=[30.75, -97.48], zoom=10,
                children=[dl.TileLayer(id="base-layer")]
            )

            admin_controls = html.Div([
                html.H3("Admin Controls"),
                dcc.Input(id='new-name', type='text', placeholder='Name'),
                dcc.Input(id='new-type', type='text', placeholder='Animal Type'),
                html.Button('Add Animal', id='add-animal')
            ]) if role == 'admin' else html.Div()

            dashboard_content = html.Div([
                html.H2(f"Welcome {username} ({role})"),
                data_table,
                pie_chart,
                map_placeholder,
                admin_controls
            ])

            return (f"Login successful as {role}.", {'display': 'block'}, dashboard_content)

        return ("Invalid credentials. Please try again.", {'display': 'none'}, None)
    return ("", {'display': 'none'}, None)

if __name__ == '__main__':
    app.run_server(debug=True)


Dash app running on http://127.0.0.1:30801/
